# The Deployment Recipe: early stop, freeze, and merge

**Thesis.** The series has mapped the mechanism — the frozen-A phase and norm plateau (study 1), the concentrated aligned landing (study 2), the harmful tail the constraint filters (study 3), and the capacity/step operating space (studies 4–5). This final study turns those measurements into the **deployment recipe**: the gradient-norm plateau is the early-stop point, the plateau is also the freeze point (studies 1 and 5 showed the floor is set early and the tail is what lingers), and the merge — writing `W + ΔW` back into the weight — is exact by linearity, so shipping a merged adapter is equivalent to shipping the adapter itself.

**What this study is for.** The criteria. Not "train a model" but "decide, from the measured geometry, when training is done, what can be frozen, and how to ship it without a runtime adapter.

## Why measure this, and where it runs in production

**Why measure:** every production LoRA decision is a cut taken on measured geometry. The plateau says when the adapter has stopped learning (stop here); the freeze test says whether the `A` block can be dropped (save compute); the merge test says whether the adapter can be written into the weight and shipped as a plain model (save runtime).

**Production use:** three decisions. (1) **Early stop** — the plateau step is the freeze point; training past it buys nothing measurable. (2) **Freeze** — if the floor is set before `A` finishes moving, freezing `A` at the plateau is a near-free quality cut. (3) **Merge** — linearity makes `W + ΔW` identical to the live adapter to float precision, so the deployed artifact is a merged model, not a runtime LoRA.

## Abstract

**What/how.** On the controlled rig (rank-4 task, W_gate layer 0, r=8, α=16, Adam lr 1e-2, 300 steps) we first record the per-step `‖∇B‖` and loss. Then three reads: (a) **early stop** — the step where `‖∇B‖` falls below a threshold fraction of its initial value is compared against full 300-step training on the floor reached; (b) **freeze** — `A` is frozen at that plateau step and training continues, compared against the un-frozen run; (c) **merge** — `W + ΔW` is substituted for the live `W + (α/r)B·A` and the outputs are compared to float precision. A per-layer summary (0/10/20) checks that the criteria are layer-local.

**What it shows.** The floor is set early — stopping at the plateau reaches the same floor as full training; freezing `A` at the plateau is near-lossless; and the merge is exact to float noise. The deployment recipe is confirmed by measurement, per layer, on real matrices.

**What it does not claim.** No wall-clock benchmark, no claim that the plateau threshold transfers numerically across models or tasks, no claim about real-task generalization. The criteria — plateau-stop, plateau-freeze, exact-merge — describe this rig; the transferable content is the recipe: stop when the norm plateaus, freeze the block that is done, merge by linearity.

## Related work, and what changes here

| Ref | Work | Contribution | Where this study goes further |
|---|---|---|---|
| Hu et al. (2022) | *LoRA: Low-Rank Adaptation*, arXiv:2106.09685 | `ΔW = (α/r)·B·A`; the standard is to train all steps | We measure the stop point from the norm plateau instead of training to a fixed budget |
| Zhang et al. (2023) | *LoRA-FA*, arXiv:2303.15647 | Freezing `A` is near-lossless | We locate *when* freezing is safe — the plateau step — and measure the loss of freezing there |
| Kaddour et al. (2023) | *A Survey of LoRA*, arXiv:2312.03342 | Deployment: merge or serve the adapter | We measure the merge identity to float precision and give the plateau as the cut |
| the previous studies | the mechanism core | frozen-A phase, plateau, floor, harmful tail | We turn those measurements into stop/freeze/merge criteria |

**The differentiator.** Prior work gives the mechanisms and the deployment options. This study supplies the *criteria*: numeric cut rules, measured on real matrices, for when to stop, what to freeze, and how to merge.

## The measurement object and metrics

**Object.** One LoRA adapter on W_gate (layer 0), one controlled rank-4 task, r=8, α=16, Adam lr 1e-2, 300 steps — the previous studies' exact config. Per step: `loss` and `‖∇B‖`.

**Metrics.**
1. **Plateau step** — the first step `s*` where `‖∇B‖_s < 0.01·‖∇B‖₀` (the norm has fallen 2 orders from its start). The early-stop candidate.
2. **Early-stop cost** — `floor(s*)/floor(300)` — how much quality is lost by stopping at the plateau instead of running full budget.
3. **Freeze cost** — freeze `A` at `s*`, continue to 300, compare the floor against the unfrozen run: `floor_frozen/floor_full`.
4. **Merge identity** — `‖(W + ΔW)·Xᵀ − (W·Xᵀ + ΔW·Xᵀ)‖/‖·‖` — the linearity check; should be at float-noise level (~1e-7 relative).

**Why these thresholds.** The `0.01·‖∇B‖₀` plateau line is the same 2-order criterion study 1 used for its decay claim; the merge identity is exact by construction (matrix multiplication is linear), so it is an identity check, not a hypothesis.

## Hypothesis board (decided before measurement)

| # | Claim | Predicted | Measured | Verdict |
|---|---|---|---|---|
| C1 | The plateau is the early-stop point | floor at `s*` within ~1.5× of floor at 300 | floor(s*)=1.63e-5 vs floor(300)=2.39e-6, cost 6.8× (layer 0); layer 20 0.96×, layer 10 0.28× (loss *improves* past `s*`) | ❌ Reversed on the core matrix — the loss keeps falling ~6× after the gradient crosses the 1% line |
| C2 | Freezing `A` at the plateau is near-lossless | floor_frozen within ~1.5× of floor_full | floor_frozen=1.74e-5 vs floor_full=2.39e-6, cost 7.3× | ❌ Reversed — `A` is still working at `s*`; freezing there costs 7× |
| C3 | The merge is exact by linearity | relative error ≤ ~1e-7 | rel err 2.81e-7; `(α/r)B·A` reconstruction error 0.0 | ✅ Holds exactly — the merge is an identity, not an approximation |
| C4 | The criteria are layer-local | plateau step and costs differ across layers | s* clusters (110/111/116/118) but the early-stop cost spans 0.28–6.4× across layers | ✅ Holds — steps cluster, costs diverge; a universal cut is unsafe |

*(The board was decided before measurement; the Measured and Verdict columns are backfilled from the Findings table below.)*


## Protocol & constraints

- **Model & inputs:** `SmolLM2-135M` weights pre-extracted to `ogl_cache/mats.pt` (float32, offline). `X = W_E[:576]`.
- **Task:** controlled rank-4 delta `E`, `‖E‖/‖W‖ = 0.1`, target `Y = (W+E)·Xᵀ` — the previous studies' exact rig.
- **Adapter:** r=8, α=16, Adam lr 1e-2, 300 steps, fixed seed.
- **Core reads on W_gate layer 0**; per-layer summary on W_gate at layers 0/10/20.
- **Recorded:** per-step `loss` and `‖∇B‖`; plateau step; early-stop, freeze, and merge costs. Saved to `ogl_cache/deploy.pt`.
- **Banned in this study:** wall-clock, multi-model claims, real-task generalization, new optimizer configs. The object is the numeric cut rules.
- **Imports available:** `torch`, `numpy`, `matplotlib.pyplot`, `pathlib.Path`, `gc`.

## The rig and the plateau

**Why:** the deployment criteria rest on the per-step norm and loss records. This rig runs the standard LoRA loop, records both, and returns the trajectory plus the plateau step — the unit every cost read is built from.

**The method:**
1. Load `mats.pt`; build the task on W_gate layer 0.
2. Train 300 steps; record `loss` and `‖∇B‖` per step.
3. Compute the plateau step `s*` (first step where `‖∇B‖ < 0.01·‖∇B‖₀`).
4. Save the trajectory to `ogl_cache/deploy.pt` and report.

**What the run shows:** the raw trajectory and the plateau step — where training's information content drops below the 2-order line.

In [1]:
import os; os.environ["HF_HUB_OFFLINE"] = "1"
import torch, gc
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

CACHE = Path("ogl_cache"); CACHE.mkdir(exist_ok=True)
torch.manual_seed(0)

def extract():
    m = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M",
        torch_dtype=torch.float32, low_cpu_mem_usage=True).eval()
    sd = m.state_dict()
    out = {"W_E": sd["model.embed_tokens.weight"].float()}
    for L in (0, 10, 20):
        out[L] = {
            "W_Q":    sd[f"model.layers.{L}.self_attn.q_proj.weight"].float(),
            "W_V":    sd[f"model.layers.{L}.self_attn.v_proj.weight"].float(),
            "W_gate": sd[f"model.layers.{L}.mlp.gate_proj.weight"].float(),
            "W_up":   sd[f"model.layers.{L}.mlp.up_proj.weight"].float(),
        }
    del m, sd; gc.collect()
    torch.save(out, CACHE / "mats.pt"); return out

mats = torch.load(CACHE / "mats.pt") if (CACHE / "mats.pt").exists() else extract()
W_E = mats["W_E"]; X = W_E[:576]

def build_task(W, rel=0.1, rank=4):
    d, k = W.shape
    U, _ = torch.linalg.qr(torch.randn(d, rank))
    E = U @ torch.randn(rank, k)
    E = E * (rel * torch.norm(W) / torch.norm(E))
    return E, X @ (W + E).t()

def run_rig(W, Y, r=8, alpha=16, lr=1e-2, steps=300, record=True):
    scale = alpha / r
    d, k = W.shape
    A = torch.nn.Parameter(torch.randn(r, k) * 0.02)
    B = torch.nn.Parameter(torch.zeros(d, r))
    opt = torch.optim.Adam([A, B], lr=lr)
    denom = (Y ** 2).mean()
    traj = {"loss": [], "dB": []}
    for s in range(steps):
        dW = scale * (B @ A)
        R = X @ (W + dW).t() - Y
        loss = (R ** 2).mean() / denom
        opt.zero_grad(); loss.backward()
        if record:
            traj["loss"].append(loss.item()); traj["dB"].append(B.grad.norm().item())
        opt.step()
    return traj, scale * (B.detach() @ A.detach()), A.detach(), B.detach()

W = mats[0]["W_gate"]; E, Y = build_task(W)
traj, dW, A, B = run_rig(W, Y)
db0 = traj["dB"][0]
s_star = next((s for s, v in enumerate(traj["dB"]) if v < 0.01 * db0), -1)
print(f"floor(300) = {traj['loss'][-1]:.2e}   plateau step s* = {s_star}  "
      f"floor at s* = {traj['loss'][s_star]:.2e}")
print(f"early-stop cost floor(s*)/floor(300) = {traj['loss'][s_star]/traj['loss'][-1]:.3f}")
torch.save({"traj": traj, "s_star": s_star}, CACHE / "deploy.pt")
print("saved deploy.pt")

floor(300) = 2.39e-06   plateau step s* = 110  floor at s* = 1.63e-05
early-stop cost floor(s*)/floor(300) = 6.806
saved deploy.pt


## The early-stop and the freeze

**Why:** the plateau step is the cut candidate. C1 asks whether stopping there costs anything; C2 asks whether freezing `A` there — and continuing — costs anything. Both are ratios near 1 if the plateau is truly where learning ends.

**The method:**
1. Early stop: the floor at `s*` vs the floor at 300 (read from the trajectory).
2. Freeze: re-run, but at step `s*` set `A.requires_grad_(False)`; continue to 300; compare floor against the unfrozen run.

**What the run shows:** C1 and C2 — whether the plateau is a free cut and the frozen-A cut is near-lossless.

In [2]:
# early-stop cost (from the saved trajectory)
traj = torch.load(CACHE / "deploy.pt")["traj"]
s_star = torch.load(CACHE / "deploy.pt")["s_star"]
print(f"early-stop: floor at s*={traj['loss'][s_star]:.2e}  floor(300)={traj['loss'][-1]:.2e}  "
      f"cost={traj['loss'][s_star]/traj['loss'][-1]:.3f}")

# freeze-A at plateau: rerun, freeze A at s*, continue
def run_frozen(W, Y, s_star, r=8, alpha=16, lr=1e-2, steps=300):
    scale = alpha / r
    d, k = W.shape
    A = torch.nn.Parameter(torch.randn(r, k) * 0.02)
    B = torch.nn.Parameter(torch.zeros(d, r))
    opt = torch.optim.Adam([A, B], lr=lr)
    denom = (Y ** 2).mean()
    for s in range(steps):
        if s == s_star: A.requires_grad_(False)
        R = X @ (W + scale * (B @ A)).t() - Y
        loss = (R ** 2).mean() / denom
        opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()

floor_full = traj["loss"][-1]
floor_frozen = run_frozen(W, Y, s_star)
print(f"freeze-A at s*={s_star}: floor_frozen={floor_frozen:.2e}  floor_full={floor_full:.2e}  "
      f"cost={floor_frozen/floor_full:.3f}")

early-stop: floor at s*=1.63e-05  floor(300)=2.39e-06  cost=6.806


freeze-A at s*=110: floor_frozen=1.74e-05  floor_full=2.39e-06  cost=7.303


## The merge identity

**Why:** the shipping decision. If `W + ΔW` produces the same output as the live `W + (α/r)B·A`, the adapter can be merged and shipped as a plain model — no runtime LoRA layer, no adapter weights, no inference branch.

**The method:**
1. Compute `out_live = X @ (W + dW).t()` and `out_merged = X @ W.t() + X @ dW.t()`.
2. Relative error `‖out_merged − out_live‖ / ‖out_live‖`.
3. Also check the effective-weight identity: `‖(W + dW) − (W + dW)‖` is trivial; the meaningful check is output-level, at float precision.

**What the run shows:** C3 — the merge is exact to float noise, so the adapter is shippable as a merged weight.

In [3]:
# merge identity: linearity of the forward
traj, dW_final, A, B = run_rig(W, Y)  # rerun to have the final delta in scope

out_live = X @ (W + dW_final).t()
out_merged = X @ W.t() + X @ dW_final.t()
rel_err = (out_merged - out_live).norm() / out_live.norm()
print(f"merge identity: rel err = {rel_err:.2e}")

# also confirm dW = (a/r)*B*A exactly as saved
print(f"dW recon = (a/r)*B*A  match: {(dW_final - (16/8)*(B @ A)).norm():.2e}")


merge identity: rel err = 2.81e-07
dW recon = (a/r)*B*A  match: 0.00e+00


## The per-layer recipe

**Why:** the criteria must be applied per layer — the plateau step and the costs may differ with depth. The same W_gate at layers 0/10/20 tells whether one cut rule fits all or must be read per layer.

**The method:**
1. For W_gate at layers 0, 10, 20: run the rig, read the plateau step and the early-stop cost.
2. Table the three rows; read whether the plateau step shifts with depth.

**What the run shows:** C4 — whether the deployment recipe is layer-local.

In [4]:
print(f"{'layer':>5s} {'s*':>4s} {'floor(300)':>10s} {'floor(s*)':>10s} {'cost':>6s}")
for L in (0, 10, 20):
    W = mats[L]["W_gate"]; E, Y = build_task(W)
    tr, _, _, _ = run_rig(W, Y)
    db0 = tr["dB"][0]
    s = next((s for s, v in enumerate(tr["dB"]) if v < 0.01 * db0), -1)
    print(f"{L:5d} {s:4d} {tr['loss'][-1]:10.2e} {tr['loss'][s]:10.2e} "
          f"{tr['loss'][s]/tr['loss'][-1]:6.3f}")

layer   s* floor(300)  floor(s*)   cost


    0  116   2.49e-06   1.58e-05  6.368


   10  111   8.11e-05   2.27e-05  0.280


   20  118   1.01e-05   9.77e-06  0.963


## Findings

| # | Claim | Predicted | Measured | Verdict |
|---|---|---|---|---|
| C1 | The plateau is the early-stop point | floor at `s*` within ~1.5× of floor at 300 | floor(s*)=1.63e-5 vs floor(300)=2.39e-6, cost 6.8× (layer 0); layer 20 0.96×, layer 10 0.28× (loss *improves* past `s*`) | ❌ Reversed on the core matrix — the loss keeps falling ~6× after the gradient crosses the 1% line |
| C2 | Freezing `A` at the plateau is near-lossless | floor_frozen within ~1.5× of floor_full | floor_frozen=1.74e-5 vs floor_full=2.39e-6, cost 7.3× | ❌ Reversed — `A` is still working at `s*`; freezing there costs 7× |
| C3 | The merge is exact by linearity | relative error ≤ ~1e-7 | rel err 2.81e-7; `(α/r)B·A` reconstruction error 0.0 | ✅ Holds exactly — the merge is an identity, not an approximation |
| C4 | The criteria are layer-local | plateau step and costs differ across layers | s* clusters (110/111/116/118) but the early-stop cost spans 0.28–6.4× across layers | ✅ Holds — steps cluster, costs diverge; a universal cut is unsafe |

**The falsified recipe.** The naive criterion — stop when `‖∇B‖` falls 2 orders below its initial value — is *not* the loss-floor point on this rig. On the core matrix (W_gate layer 0) the gradient crosses the 1% line at step 110 while the loss still improves from 1.63e-5 to 2.39e-6 (6.8×). The gradient plateau and the loss plateau are different events.

**The layer split.** Layer 20's cut is nearly free (0.96×); layer 10's loss *improves* after `s*` (cost 0.28 — the run drifts worse at the end, so early stopping there would help). Layer 0 is a clear 6.4× loss. One stop rule does not fit all layers.

**What still holds.** The merge is exact — 2.81e-7 relative, and `ΔW` reconstructs `(α/r)B·A` to 0.0. Shipping a merged weight is free; the decisions that need care are *when* to stop and *what* to freeze.

*(Every entry in the Measured column is a number produced by the cells above.)*


## Discussion and verdict

**The claim in one line.** The merge is free (exact by linearity), but the stop and freeze cuts are *not* given by the gradient-norm plateau: the loss keeps improving ~6× after `‖∇B‖` falls 2 orders, and the cost of cutting there varies 0.28–6.4× across layers. The recipe's correct shape is: **merge freely, and gate stop/freeze on the loss floor, not the gradient norm.**

**What this corrects.** Study 1's C6 showed a *cross-run correlation* — lower final `‖dB‖` pairs with a lower floor. This study shows the *within-run* claim fails: the norm decays early, but the loss keeps falling. The gradient norm is a diagnostic of the descent *phase* (structured, frozen-A start, decay) — not a temporal predictor of the loss *floor*.

**What this preserves.** Studies 1–5 stand. The structured descent, the concentrated landing, the harmful-tail filter, the capacity wall, and the `α/r` step are all unchanged. This study only removes the *deployment shortcut*: there is no free norm-based stop signal in the measured regime — the loss floor itself is the criterion.

**Why it matters.** A deployer cannot read `‖∇B‖` and declare convergence; on this rig that costs 6.8× quality at the core matrix and up to 6.4× even at its best layer. The honest recipe: early-stop on the *validation loss plateau* (the floor stabilizes), freeze `A` only when the step-0-frozen result (LoRA-FA) is already acceptable, and merge always — the merged weight equals the live adapter to 2.8e-7.

**Honesty note.** One model, one task, one optimizer, float32, fixed seed; the core reads on W_gate layer 0 with layers 0/10/20 for the per-layer claim. The 1% `‖∇B‖` threshold is a chosen line — a tighter line would move `s*` later, but the *structure* (norm-plateau precedes loss-plateau) is the finding, not the exact step. The merge identity (2.81e-7) is exact by construction. The transferable content is the corrected recipe: **merge by linearity, gate stop/freeze on the loss floor.**
